# 05. 파이프라인 — 공통 정의를 모듈로 옮기고, 여기서 실험한다

## 왜 노트북을 나눴나

`04_model_selection.ipynb`가 **229셀**까지 길어져 읽기 어려워졌고, 더 큰 문제가 있었다:
**커널을 새로 켤 때마다 정의 셀 15개를 순서대로 실행**해야 했고, 한 번은 예측 부분을 저장하기 전에
커널이 초기화돼 **51번을 다시 학습**해야 했다.

그래서 검증을 마친 정의를 전부 `src/pipeline.py`로 옮겼다. 이 노트북은 그것을 import 해서 쓴다.

| 파일 | 역할 |
|---|---|
| `src/metric.py` | 대회 공식 산식 (수정 금지) |
| `src/submission.py` | 제출 파일 생성·검증 |
| **`src/pipeline.py`** | **공통 정의 전부** — 데이터·fold·피처·풍속·라벨·LightGBM·채점 |
| `src/nn.py` | 신경망 — 산식손실 MLP(`MetricMLP`) + 풍속 MLP(`WindMLP`) |
| `notebooks/04_model_selection.ipynb` | **동결.** 1~21절 실험 기록 (출력 포함) |
| **`notebooks/05_pipeline.ipynb`** | **여기.** 앞으로의 실험 |
| `notebooks/train.ipynb` / `inference.ipynb` | 2차 평가용. **같은 `pipeline`을 import** (~2026-08-10) |

`04`는 지우지 않는다. 실행 출력이 다 붙어 있는 **실험 원장**이라 결론의 근거가 거기 있다.

---

## 현재 위치 (2026-08-03)

| 제출 | 구성 | Score | 1-NMAE | FICR |
|---|---|---|---|---|
| v4 | LGB `B_norm` τ=0.50 | 0.6315 | 0.8620 | 0.4009 |
| **v5** | **+ 산식손실 MLP 0.3** | **0.6413** | **0.8671** | **0.4155** |
| **1등** | | **0.67365** | **0.87964** | **0.46767** |

**격차 -0.0323의 81%가 FICR이고, 그 FICR 격차의 98%가 "오차 10% 감소" 하나로 설명된다.**
1등은 산식 트릭을 쓴 게 아니라 더 정확하다. 우리 σ의 89%는 예보 오차다(04 노트북 14절).

## 확정된 구성

```
1단계  풍속 2종(GBDT l2 결정적 · MLP)  →  각각 LightGBM 발전량 예측  →  0.7 : 0.3
2단계  위 결과 (1-w)  +  산식손실 MLP  w                              →  최종
       LightGBM: B_norm 라벨, τ=0.50, actual 가중, 상위 200피처, seed 5
       산식손실 MLP: 원본 라벨, 채점행만, T_SOFT=0.006, full-batch, seed 5
```

## ⚠️ 검증 지표 규칙 (v5 리더보드가 알려준 것)

- **LightGBM 계열**(피처·하이퍼파라미터) → **B안 평균**
- **신경망 계열**(블렌드 비중·구조) → **A안**

B안 fold는 학습 구간이 1.5~3년이라 **데이터를 많이 먹는 모델을 구조적으로 과소평가**한다.
최종 모델은 3년 전체를 쓴다. 실제로 v5의 로컬 예측은 +0.0021이었으나 리더보드는 **+0.0098**이었다.

---

## 1. 셋업 + 이식 검증

`src/pipeline.py`가 04 노트북의 정의를 **정확히** 옮겼는지 확인한다.
아래 두 값이 소수점 넷째 자리까지 재현되면 통과다 (LightGBM은 결정적이라 정확히 같아야 한다).

| 구성 | A안(2024) | B안 평균 |
|---|---|---|
| `B_norm` τ=0.50 (v4 발전량 모델) | **0.6418** | **0.6356** |
| `A_asis` τ=0.60 (v2 구성) | **0.6391** | **0.6298** |

⏱️ 풍속 12회 + LightGBM 24회 = **36회 학습** (약 5분).

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
import torch

import src.pipeline as pl
import src.nn as mnn
from src.metric import CAPACITY_KWH, TARGET_COLS

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print("저장소 루트:", pl.REPO_ROOT)
ctx = pl.load_context(with_test=True, with_avail=True)
print("train:", ctx.train.shape, "| test:", ctx.test.shape)
print("공통 피처:", len(ctx.common_cols), "| 그룹 전용:", {g: len(v) for g, v in ctx.group_cols.items()})
print("fold:", list(ctx.fold_info))

print("\n=== 가동률 요약 (04 노트북 20-1과 같아야 정상) ===")
rows = []
for g in pl.GROUP_COLS:
    a, lab = ctx.avail[g], ctx.train[g].notna()
    both = lab & a.notna()
    n_t = len(pl.TURBINES[g])
    rows.append({"그룹": g, "커버리지": round(both.sum() / lab.sum(), 4),
                 "평균 가동대수": round(a[both].mean() * n_t, 2),
                 "전대수 가동": round((a[both] >= 1 - 1e-9).mean(), 4)})
display(pd.DataFrame(rows).set_index("그룹"))
print("기대값: 커버리지 1.0/1.0/0.9647, 평균가동 5.81/5.84/4.81, 전대수 0.8435/0.8527/0.7719")

**확인할 것**: 가동률 요약이 기대값과 일치하는지. 다르면 `compute_availability` 이식에 문제가 있는 것입니다.

In [ ]:
# ── 이식 검증: 04 노트북의 두 기준선을 재현한다 (풍속 12 + LGB 24 = 36회) ──
PRED_CACHE = {}
stage = []

print("=== LGB_bnorm_q50 (v4 발전량 모델) ===")
stage.append(pl.run_variant(ctx, PRED_CACHE, "LGB_bnorm_q50",
                            pl.lgbm_fold_fit_fn(mode="B_norm", tau=0.50)))
print("\n=== LGB_asis_q60 (v2 구성) ===")
stage.append(pl.run_variant(ctx, PRED_CACHE, "LGB_asis_q60",
                            pl.lgbm_fold_fit_fn(mode="A_asis", tau=0.60)))

print("\n" + "=" * 70)
tbl = pl.summarize(stage)
display(tbl)

EXPECT = {"LGB_bnorm_q50": (0.6418, 0.6356), "LGB_asis_q60": (0.6391, 0.6298)}
ok = True
for name, (a_exp, b_exp) in EXPECT.items():
    a_got, b_got = tbl.loc[name, "A안(2024)"], tbl.loc[name, "B안 평균"]
    good = abs(a_got - a_exp) < 5e-4 and abs(b_got - b_exp) < 5e-4
    ok &= good
    print(f"  {name}: A안 {a_got:.4f} (기대 {a_exp}) / B평균 {b_got:.4f} (기대 {b_exp})  "
          f"{'✓' if good else '✗ 불일치!'}")
print("\n" + ("✅ 이식 검증 통과 — src/pipeline.py를 신뢰할 수 있다"
              if ok else "❌ 불일치. 여기서 멈추고 원인을 찾을 것"))

**확인할 것**: **✅ 이식 검증 통과**가 떠야 합니다. 불일치가 뜨면 멈추고 알려주세요.

---

## 2. 최종 파이프라인 — test 예측 부분 만들기

`lgb_part`(LightGBM 부분)와 `mlp_part`(산식손실 MLP 부분)를 만들어
**`models/v5_parts.npz`에 저장**합니다. 한 번 저장해두면 커널이 초기화돼도
블렌드 비중 실험을 **학습 0회**로 다시 할 수 있습니다.

⏱️ 풍속 6회 + LightGBM 30회 + 산식손실 MLP 15회 = **51회** (약 15분).
저장 파일이 이미 있으면 건너뜁니다.

In [ ]:
import src.submission as subm
from src.submission import build_submission, validate_submission, save_submission

SAMPLE_PATH = pl.REPO_ROOT / "data" / "sample_submission.csv"
subm.SAMPLE_SUBMISSION_PATH = SAMPLE_PATH
subm.SUBMISSIONS_DIR = pl.REPO_ROOT / "submissions"    # ⚠️ 둘 다 덮어써야 한다 (17-4 함정)
PARTS_PATH = pl.REPO_ROOT / "models" / "v5_parts.npz"
PARTS_PATH.parent.mkdir(parents=True, exist_ok=True)

FINAL_SEEDS = [pl.SEED, 7, 123, 2024, 31]
WIND_BLEND = {"gbdt": 0.7, "mlp": 0.3}     # 1단계: 풍속 2종 (18-5)
T_SOFT = 0.006                             # 21-3 스윕
TAU, LABEL_MODE = pl.DEFAULT_TAU, pl.DEFAULT_LABEL_MODE

_gaps = ctx.test["kst_dtm"].diff().dropna().unique()
assert ctx.test["kst_dtm"].is_monotonic_increasing and len(_gaps) == 1 and _gaps[0] == pd.Timedelta("1h")
print("✓ test 시간축 1시간 연속")


def build_final_parts():
    """전체 train으로 학습해 test 예측의 두 부분을 만든다."""
    full = ctx.full_mask
    ws_final = {}

    print("\n=== 풍속 GBDT (결정적) ===")
    for g in pl.GROUP_COLS:
        m = pl.fit_wind_gbdt(ctx, g, full, "l2")
        Xtr = pl.build_wind_input_frame(ctx, ctx.train, g)
        Xte = pl.build_wind_input_frame(ctx, ctx.test, g)
        assert list(Xtr.columns) == list(Xte.columns), f"{g}: 풍속 입력 컬럼 불일치"
        ws_final[("gbdt", g)] = (pd.Series(m.predict(Xtr), index=ctx.train.index).clip(lower=0.0),
                                 pd.Series(m.predict(Xte), index=ctx.test.index).clip(lower=0.0))
        print(f"  {g}: train 내 상관 {ws_final[('gbdt', g)][0].corr(ctx.train[f'scada_ws_{g}']):.4f}")

    print("\n=== 풍속 MLP ===")
    for g in pl.GROUP_COLS:
        Xtr = pl.build_wind_input_frame(ctx, ctx.train, g)
        Xte = pl.build_wind_input_frame(ctx, ctx.test, g)
        y = ctx.train[f"scada_ws_{g}"]
        fit_idx = y.notna()
        cut = ctx.train.loc[fit_idx, "kst_dtm"].quantile(0.9)
        tr = fit_idx & (ctx.train["kst_dtm"] <= cut)
        es = fit_idx & (ctx.train["kst_dtm"] > cut)
        o_tr, o_te = mnn.fit_wind_mlp(Xtr, y, tr, es, [Xtr, Xte], seed=pl.SEED)
        ws_final[("mlp", g)] = (pd.Series(o_tr, index=ctx.train.index),
                                pd.Series(o_te, index=ctx.test.index))
        print(f"  {g}: train 내 상관 {ws_final[('mlp', g)][0].corr(y):.4f}")

    print("\n=== LightGBM 발전량 (풍속 2종 × seed 5) ===")
    lgb_pred, pc_keep = {}, {}
    for src_ in ["gbdt", "mlp"]:
        for g in pl.GROUP_COLS:
            ws_tr, ws_te = ws_final[(src_, g)]
            ok = ctx.train[g].notna()
            edges, vals = pl.fit_power_curve_oracle(ws_tr[ok].to_numpy(dtype=float),
                                                    ctx.train.loc[ok, g].to_numpy(dtype=float))
            tag = f"fin_{src_}"
            Xtr = pl.build_frame_given_pc(ctx, ctx.train, g, ws_tr, edges, vals, tag)
            Xte = pl.build_frame_given_pc(ctx, ctx.test, g, ws_te, edges, vals, tag)
            assert list(Xtr.columns) == list(Xte.columns), f"{src_}/{g}: 입력 컬럼 불일치"

            m0 = pl.lgbm_train_label(ctx, Xtr, g, full, TAU, pl.SEED, LABEL_MODE)
            keep = pl.select_features(m0, Xtr.columns, pl.BEST_TOPN)
            preds = [pl.lgbm_train_label(ctx, Xtr[keep], g, full, TAU, sd, LABEL_MODE, rand=True)
                     .predict(Xte[keep]) for sd in FINAL_SEEDS]
            lgb_pred[(src_, g)] = np.clip(np.mean(preds, axis=0), 0, CAPACITY_KWH[g])
            if src_ == "gbdt":
                pc_keep[g] = (edges, vals, keep)
            print(f"  ✓ {src_}/{g}: 이용률 {lgb_pred[(src_, g)].mean() / CAPACITY_KWH[g] * 100:.1f}%")

    lgb_part = {g: sum(WIND_BLEND[s] * lgb_pred[(s, g)] for s in WIND_BLEND) for g in pl.GROUP_COLS}

    print("\n=== 산식손실 MLP (GBDT 풍속, 시드 5) ===")
    mlp_part = {}
    for g in pl.GROUP_COLS:
        cap = CAPACITY_KWH[g]
        ws_tr, ws_te = ws_final[("gbdt", g)]
        edges, vals, keep = pc_keep[g]
        Xtr = pl.build_frame_given_pc(ctx, ctx.train, g, ws_tr, edges, vals, "fin_gbdt")[keep].to_numpy(float)
        Xte = pl.build_frame_given_pc(ctx, ctx.test, g, ws_te, edges, vals, "fin_gbdt")[keep].to_numpy(float)

        ratio = (ctx.train[g] / cap).to_numpy(dtype=float)
        fit_idx = ctx.train[g].notna().to_numpy() & (ratio >= mnn.EVAL_MIN_RATIO)   # 채점 대상만
        times = ctx.train["kst_dtm"].to_numpy()
        cut = pd.Series(ctx.train.loc[fit_idx, "kst_dtm"]).quantile(0.9)
        tr = fit_idx & (times <= np.datetime64(cut))
        es = fit_idx & (times > np.datetime64(cut))

        mu, sd_ = mnn.fit_standardizer(Xtr[tr])          # 표준화는 학습 구간에서만
        Xes_t = torch.tensor(((Xtr[es] - mu) / sd_).astype(np.float32))
        y_es = ratio[es]

        def _eval(model, _X=Xes_t, _y=y_es):
            with torch.no_grad():
                return mnn.group_score(_y, np.clip(model(_X).numpy(), 0.0, 1.0))

        seed_preds = []
        for s_ in FINAL_SEEDS:
            model, best_ep = mnn.train_metric_mlp((Xtr[tr] - mu) / sd_, ratio[tr],
                                                  seed=s_, t_soft=T_SOFT, eval_fn=_eval)
            seed_preds.append(mnn.predict_ratio(model, Xte, mu, sd_))
            print(f"    {g}/seed{s_}: {best_ep}에폭")
        mlp_part[g] = np.mean(seed_preds, axis=0) * cap
        print(f"  ✓ {g}: 이용률 {mlp_part[g].mean() / cap * 100:.1f}%")
    return lgb_part, mlp_part


if PARTS_PATH.exists():
    _z = np.load(PARTS_PATH)
    lgb_part = {g: _z[f"lgb_{g}"] for g in pl.GROUP_COLS}
    mlp_part = {g: _z[f"mlp_{g}"] for g in pl.GROUP_COLS}
    print("저장된 예측 부분 로드:", PARTS_PATH, "(다시 만들려면 이 파일을 지우세요)")
else:
    lgb_part, mlp_part = build_final_parts()
    np.savez(PARTS_PATH,
             **{f"lgb_{g}": lgb_part[g] for g in pl.GROUP_COLS},
             **{f"mlp_{g}": mlp_part[g] for g in pl.GROUP_COLS})
    print("\n저장:", PARTS_PATH)

for g in pl.GROUP_COLS:
    print(f"  {g}: LGB {lgb_part[g].mean() / CAPACITY_KWH[g]:.4f} / "
          f"MLP {mlp_part[g].mean() / CAPACITY_KWH[g]:.4f}")

**확인할 것**
- **`models/v5_parts.npz`가 만들어졌는지.** 이제 커널이 초기화돼도 블렌드 실험은 학습 0회입니다
- LGB 부분과 MLP 부분의 이용률이 0.05 이상 벌어지지 않는지
- **에폭 수가 `MAX_EPOCHS=400`에 붙어 있는지.** 붙어 있으면 학습이 덜 된 것이니 알려주세요

---

## 3. 블렌드 비중 재검토 — 학습 0회

v5(w=0.3)의 리더보드 오프셋이 **처음으로 양수(+0.0036)** 였습니다.
로컬이 MLP 기여를 5배 과소평가했다는 뜻이고, **최적 비중도 0.3보다 높을 가능성**이 큽니다.

| MLP 비중 | 0.0 | 0.3 | 0.5 | 0.7 | 0.8 | 1.0 |
|---|---|---|---|---|---|---|
| **A안**(학습 2년) | 0.6418 | 0.6424 | 0.6429 | 0.6431 | **0.6438** | 0.6420 |
| B평균(학습 1.5~3년) | 0.6356 | **0.6377** | 0.6377 | 0.6355 | 0.6344 | 0.6306 |

**최종 학습은 A안보다도 데이터가 많습니다.** 신경망 비중은 A안을 주 지표로 봅니다.

**0.5 → 0.7 순으로 한 단계씩** 올립니다. A안도 결국 fold 하나라 그 봉우리가 노이즈일 수 있어
한 번에 0.8로 뛰지 않습니다.

In [ ]:
V5_PATH = pl.REPO_ROOT / "submissions" / "20260802_v5_bnorm_q50_metricmlp03.csv"
prev5 = pd.read_csv(V5_PATH)
key = "forecast_kst_dtm" if "forecast_kst_dtm" in prev5.columns else prev5.columns[0]


def make_blend(w):
    """MLP 비중 w로 섞어 제출 DataFrame을 만든다."""
    pred = {g: np.clip((1 - w) * lgb_part[g] + w * mlp_part[g], 0, CAPACITY_KWH[g])
            for g in pl.GROUP_COLS}
    df = pd.DataFrame(pred, index=ctx.test.index)
    df["forecast_kst_dtm"] = ctx.test["kst_dtm"].dt.strftime("%Y-%m-%d %H:%M:%S")
    sub = build_submission(df, sample_path=SAMPLE_PATH)
    validate_submission(sub, sample_path=SAMPLE_PATH)
    return sub


SUBS, rows = {}, []
for w in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    sub = SUBS[w] = make_blend(w)
    r = {"MLP 비중": w}
    for g in pl.GROUP_COLS:
        r[g.split("_")[-1] + " 이용률"] = round(sub[g].mean() / CAPACITY_KWH[g], 4)
    a = np.concatenate([prev5[g].to_numpy(float) for g in pl.GROUP_COLS])
    b = np.concatenate([sub[g].to_numpy(float) for g in pl.GROUP_COLS])
    r["상관(v5)"] = round(float(np.corrcoef(a, b)[0, 1]), 4)
    rows.append(r)

print("=== 비중별 제출 후보 ===")
display(pd.DataFrame(rows).set_index("MLP 비중"))

_d = max(abs(SUBS[0.3][g].to_numpy(float) - prev5[g].to_numpy(float)).max() for g in pl.GROUP_COLS)
print(f"\nw=0.3 재현 오차: {_d:,.1f} kWh")
print("★ 0에 가까우면 pipeline 이식과 재블렌드가 모두 옳다 (제출한 v5와 동일).")
print("  0이 아니면 파워커브·피처선택·시드 중 하나가 04와 달라진 것이니 원인을 찾을 것.")

# ── 확인 후 저장 (오늘 남은 제출 횟수만큼) ─────────────────────────
# print(save_submission(SUBS[0.5], "20260803_v6_metricmlp05.csv", sample_path=SAMPLE_PATH))
# print(save_submission(SUBS[0.7], "20260803_v7_metricmlp07.csv", sample_path=SAMPLE_PATH))

---

## 4. 산식손실 MLP 튜닝 — 여태 `T_SOFT`만 건드렸다

### 지금까지 무엇이 튜닝됐고 무엇이 안 됐나 (정직한 재고 조사)

| 대상 | 상태 |
|---|---|
| 풍속 LightGBM | ✅ 5방향 시도 → **전부 기본값을 못 이김** (04의 18-2/18-3). 유용한 negative result |
| 발전량 LightGBM | ⚠️ 손실·τ·표본가중·피처수만. **`learning_rate`·`num_leaves`·`min_child_samples`·정규화는 기본값** |
| **산식손실 MLP** | ❌ **`T_SOFT`만 스윕(21-3).** 나머지는 전부 이전 프로젝트가 **179피처로** 정한 값 |
| 풍속 MLP | ❌ 없음 (512-256-128, 18-4에서 임의 결정) |
| 블렌드 비중 | ✅ 풍속 0.7/0.3, 최종 0.3 (재검토 중 — 3절) |

### 왜 MLP부터인가

1. **v5에서 MLP가 리더보드 +0.0098을 만들었다.** 이제 주력 모델이다
2. **값이 남의 것이다.** 179피처·다른 풍속·다른 라벨 처리에서 정해진 값이 우리에게 최적일 이유가 없다
3. **비용이 싸다.** MLP 1회 10초 → 한 설정을 전 fold 검증하는 데 12회 = **약 2분**.
   LightGBM(1회 30초+)보다 훨씬 싸서 여러 설정을 감당할 수 있다

### 가장 의심스러운 것 — 학습량

**full-batch라 1에폭 = 기울기 1스텝**이다. 그런데 21-2 시험 학습이 **61에폭**에서 최적을 찍었다.
즉 **기울기 갱신 61번**으로 학습이 끝났다. 파라미터 12만 개짜리 모델치고는 매우 적다.
`PATIENCE=60`이 너무 빨리 잘랐을 가능성이 크다.

### 탐색 방식 — 좌표별(one-factor-at-a-time), 격자 아님

전체 격자를 돌리면 설정이 수백 개가 되고 **다중비교로 가짜 승자**가 나온다.
근거가 강한 순서대로 **한 번에 하나씩** 바꾸고, 이긴 것만 다음 단계로 물려준다.

| 단계 | 바꾸는 것 | 근거 |
|---|---|---|
| 4-1 | **학습량** (`patience`, `max_epochs`) | 위 — 기울기 61스텝은 너무 적다 |
| 4-2 | **피처 세트** (200 / 400 / 전체) | **트리 중요도로 고른 200개가 매끄러운 모델에 맞을 이유가 없다** |
| 4-3 | **구조** (폭·깊이) | 표본 1.5만에 맞는 크기를 우리 데이터로 확인 |
| 4-4 | **정규화** (dropout, weight decay) | 구조를 키웠다면 함께 조정해야 한다 |

### ⚠️ 판정 규칙

- **주 지표는 `A안`** (신경망 계열 규칙). 단 **`B안 평균`이 크게 나빠지면 채택하지 않는다**
- 여기서는 **시드 1개**로 비교한다(빠르게). 시드 간 σ가 0.0024였으므로
  **A안 개선이 +0.005 미만이면 노이즈로 본다.** 최종 후보만 시드 5개로 재확인
- 매 단계 **기준선(현행 설정)을 같이 돌려** 같은 조건에서 비교한다

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-1. 학습량 — 기울기 스텝이 61번밖에 안 됐다
# ══════════════════════════════════════════════════════════════════
import time

MLP_CACHE = {}          # 이 절 전용 예측 캐시 (fold별 점수표는 stage_mlp에 쌓는다)
stage_mlp = []


def try_cfg(label, cfg, seeds=(pl.SEED,)):
    """설정 하나를 전 fold에서 돌리고 점수표를 돌려준다."""
    t0 = time.time()
    print(f"=== {label} ===  {cfg}")
    d = pl.run_variant(ctx, MLP_CACHE, label, pl.metric_mlp_fit_fn(cfg, seeds=seeds))
    stage_mlp.append(d)
    print(f"    ({time.time() - t0:.0f}초)")
    return d


def show(*extra):
    """A안을 주 지표로, B안 평균을 보조로 나란히."""
    t = pl.summarize(stage_mlp + list(extra))
    return t.sort_values("A안(2024)", ascending=False)


def epochs_used(label, cfg, seed=pl.SEED):
    """그 설정이 실제로 몇 에폭을 썼는지 (조기 종료가 언제 걸렸나)."""
    out = {}
    for fold, info in ctx.fold_info.items():
        for g in pl.GROUP_COLS:
            _, ep = pl.metric_mlp_predict(ctx, g, info["cv_suffix"], info["train_mask"], cfg, seed)
            out[(fold.split("(")[0].strip(), g.split("_")[-1])] = ep
    return out


# ── 기준선 (현행 = 21절에서 쓴 설정) ───────────────────────────────
BASE = dict(pl.MLP_CFG)
try_cfg("M_base", BASE)
print("\n기준선이 쓴 에폭 수:", epochs_used("M_base", BASE))
print("★ 대부분 max_epochs(400)에 한참 못 미치면 patience가 일찍 자른 것이다.")

# ── 학습량을 늘려 본다 ─────────────────────────────────────────────
try_cfg("M_pat120", {**BASE, "patience": 120})
try_cfg("M_pat120_ep800", {**BASE, "patience": 120, "max_epochs": 800})

display(show())
print("""
★ 읽는 법
  · A안이 +0.005 이상 올랐으면 채택 (시드 간 σ 0.0024의 2배)
  · max_epochs를 늘렸는데 점수가 같다면 → 이미 수렴한 것. patience만 남긴다
  · B안 평균이 -0.005 이상 떨어지면 과적합 신호이므로 채택하지 않는다""")

**확인할 것**
- **에폭 수 출력.** 대부분 100 미만이면 `patience`가 일찍 잘랐다는 뜻이고, `M_pat120`이 이길 가능성이 큽니다
- 800까지 늘려도 안 늘어나면 이미 수렴한 것이니 **`patience`만 채택**하고 `max_epochs`는 400으로 둡니다

**이긴 설정을 아래 `BEST1`에 반영하고 4-2로 갑니다.**

---

### 4-2. 피처 세트 — 트리가 고른 200개가 MLP에도 맞을까

지금 MLP는 **LightGBM 중요도 상위 200개**를 씁니다. 그런데 트리와 신경망은 피처를 쓰는 방식이 다릅니다.

- **트리**: 한 번에 한 변수로 자른다 → 상관 높은 피처 중 하나만 있으면 된다
- **신경망**: 모든 입력의 선형결합을 만든다 → **약한 신호가 많이 모이면 도움이 된다.**
  트리에게 gain 0이던 피처가 신경망에는 쓸모 있을 수 있다

04의 15-2에서 "LightGBM은 top50까지 줄여도 점수가 같다"고 확인했는데, **그건 트리 얘기**입니다.

⏱️ 3설정 × 12회. 전체(850피처)는 조금 느립니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-2. 피처 세트
# ══════════════════════════════════════════════════════════════════
# 4-1에서 이긴 설정을 여기에 반영하세요 (표를 보고 직접 수정)
BEST1 = {**BASE, "patience": 120}

try_cfg("F_top200", {**BEST1, "top_n": 200})     # = BEST1과 동일 (이름만 명시)
try_cfg("F_top400", {**BEST1, "top_n": 400})
try_cfg("F_all", {**BEST1, "top_n": "all"})      # 프레임의 모든 컬럼(약 850개)

display(show())
print("""
★ 피처를 늘려 좋아지면 → '트리 중요도 선택이 MLP에 최적이 아니다'가 확인된 것.
  그렇다면 LightGBM과 MLP가 **서로 다른 피처 집합**을 쓰게 되고, 블렌드 다양성도 올라간다.
★ 나빠지면 → 200개가 맞았다. 표본 1.5만에 피처 850개는 과적합이라는 뜻.""")

**확인할 것**: 피처를 늘려 좋아지는지. 좋아지면 **LightGBM과 MLP가 서로 다른 피처를 쓰게 되고**, 그 자체로 블렌드 다양성이 올라갑니다.

---

### 4-3. 구조 — 폭과 깊이

현재 `256 → 256`. 이전 프로젝트가 **179피처**에 맞춰 정한 값입니다.
피처가 200~850개로 늘면 첫 층이 좁아 병목이 될 수 있습니다.

| 후보 | 의도 |
|---|---|
| `256, 256` | 현행 |
| `512, 512` | 폭 확대 |
| `512, 256, 128` | 깊이 추가 + 점감 (풍속 MLP가 쓰는 구조) |
| `128, 128` | 축소 — **과적합이 문제라면 이쪽이 이긴다** |

축소 후보를 넣는 이유: 확대만 시험하면 "더 큰 게 낫다"는 결론밖에 안 나옵니다. **양쪽을 봐야 어느 방향이 맞는지 알 수 있습니다.**

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-3. 구조 (폭·깊이)
# ══════════════════════════════════════════════════════════════════
# 4-2에서 이긴 top_n을 반영하세요
BEST2 = {**BEST1, "top_n": 200}

for name, hid in [("H_256x2", (256, 256)), ("H_512x2", (512, 512)),
                  ("H_512_256_128", (512, 256, 128)), ("H_128x2", (128, 128))]:
    try_cfg(name, {**BEST2, "hidden": hid})

display(show())
print("""
★ 확대가 이기면 → 표현력이 부족했던 것. 4-4에서 정규화를 함께 올려 과적합을 막는다
★ 축소가 이기면 → 과적합이었던 것. dropout·weight decay도 올려 볼 값어치가 있다
★ 차이가 0.005 미만이면 → 구조는 중요하지 않다. **가장 작은 것**을 고른다 (학습이 빠르다)""")

**확인할 것**: 차이가 +0.005 미만이면 **가장 작은 구조**를 고릅니다(학습이 빠르고 과적합 위험이 낮습니다).

---

### 4-4. 정규화 — dropout · weight decay

구조를 키웠다면 정규화도 함께 봐야 합니다. 현재 `dropout 0.15`, `weight_decay 1e-4`.

⚠️ 여기까지 오면 비교한 설정이 **13개**입니다. 다중비교 편향이 쌓입니다.
**4-4에서 이긴 설정은 반드시 4-5에서 시드 5개로 재확인**하고, 그때도 이겨야 채택합니다.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-4. 정규화
# ══════════════════════════════════════════════════════════════════
BEST3 = {**BEST2, "hidden": (256, 256)}     # 4-3에서 이긴 구조를 반영하세요

for name, kw in [("R_d015_w1e4", {}),                                   # 현행
                 ("R_d025", {"p_drop": 0.25}),
                 ("R_d005", {"p_drop": 0.05}),
                 ("R_wd1e3", {"weight_decay": 1e-3})]:
    try_cfg(name, {**BEST3, **kw})

display(show())

---

### 4-5. 최종 후보 확인 — 시드 5개 + 블렌드까지

앞에서 고른 설정을 **시드 5개**로 다시 돌리고, LightGBM과 블렌드했을 때까지 확인합니다.
**여기서 기준선(21절 설정, 리더보드 0.6413의 그 MLP)을 못 이기면 채택하지 않습니다.**

⏱️ 2설정 × 5시드 × 12 fold·그룹 = 120회, 약 20분.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# 4-5. 최종 확인 (시드 5개) + LightGBM 블렌드
# ══════════════════════════════════════════════════════════════════
MLP_SEEDS = [pl.SEED, 7, 123, 2024, 31]
BEST_CFG = dict(BEST3)          # 4-4 결과를 반영하세요
print("최종 후보:", BEST_CFG)

final_stage = []
final_stage.append(try_cfg("MLP_old_s5", BASE, seeds=MLP_SEEDS))        # 21절 설정 = v5의 MLP
final_stage.append(try_cfg("MLP_new_s5", BEST_CFG, seeds=MLP_SEEDS))    # 튜닝 결과

# LightGBM(v4 발전량 모델) 예측을 같은 캐시에 넣어 블렌드 곡선을 그린다
if ("A안(2024)", "LGB_bnorm_q50", pl.GROUP_COLS[0]) in PRED_CACHE:
    for k, v in PRED_CACHE.items():
        if k[1] == "LGB_bnorm_q50":
            MLP_CACHE[k] = v
else:
    print("⚠️ 1절의 LGB_bnorm_q50이 캐시에 없습니다. 1절을 먼저 실행하세요.")

print("\n=== LightGBM + MLP 블렌드 곡선 ===")
rows = []
for mlp_var in ["MLP_old_s5", "MLP_new_s5"]:
    for w in np.arange(0.0, 1.01, 0.1):
        a_, b_, mn_, d_ = pl.blend_score(ctx, MLP_CACHE, "LGB_bnorm_q50", mlp_var, w)
        rows.append({"MLP": mlp_var, "비중": round(w, 1), "A안": round(a_, 4),
                     "B평균": round(b_, 4), "B최솟값": round(mn_, 4)})
curve = pd.DataFrame(rows)
display(curve.pivot(index="비중", columns="MLP", values="A안").round(4))
print("★ 위 표는 A안 기준(신경망 계열 주 지표). 아래는 B안 평균.")
display(curve.pivot(index="비중", columns="MLP", values="B평균").round(4))

print("""
★ 채택 조건 (전부 만족해야)
  1. MLP_new 단독이 MLP_old 단독보다 A안에서 +0.005 이상 높다
  2. 블렌드 최고점도 MLP_new 쪽이 높다
  3. B안 평균이 크게(-0.005 이상) 나빠지지 않았다
  ⇒ 만족하면 src/pipeline.py의 MLP_CFG를 고치고, 2절의 v5_parts.npz를 지우고 다시 만든다""")

**확인할 것 — 채택 조건 3가지가 전부 만족되는가**

```
MLP_old 단독 A안: ______   MLP_new 단독 A안: ______   차이: ______  (+0.005 이상?)
블렌드 최고   old: ______   new: ______   최적 비중: ______
B안 평균 변화: ______  (-0.005보다 나쁘면 기각)
```

**채택하면 할 일** (순서대로)
1. `src/pipeline.py`의 `MLP_CFG`를 새 값으로 수정
2. **`models/v5_parts.npz`를 삭제**하고 2절을 다시 실행 (test 예측 재생성)
3. 3절에서 새 블렌드 비중으로 제출 파일 생성

**기각하면**: 현행 설정이 이미 좋았다는 뜻입니다. 바로 5절 ③번(σ 줄이기)으로 갑니다.

⚠️ **여기까지 15개 설정을 비교했습니다.** 다중비교 편향을 감안해
**A안 +0.005 미만은 개선으로 세지 않습니다.** 애매하면 **더 단순한 쪽(현행)** 을 남기세요.

**확인할 것**
- **`w=0.3 재현 오차`가 0에 가까운지.** 이게 `src/pipeline.py` 이식의 최종 검증입니다.
  (신경망은 시드 고정이라도 환경에 따라 미세하게 다를 수 있어 수십 kWh 수준이면 정상입니다.
  수천 kWh 이상 벌어지면 이식 문제입니다)
- 비중이 오를수록 이용률이 급격히(±0.03 이상) 변하지 않는지

```
v6 (w=0.5)  Score: ______  1-NMAE: ______  FICR: ______
v7 (w=0.7)  Score: ______  1-NMAE: ______  FICR: ______
  기준: v5 (w=0.3) = 0.6413 / 0.8671 / 0.4155
```

---

## 5. 다음 — 1등(0.67365)까지의 로드맵

### ① 블렌드 비중 (위 3절) — 학습 0회, 기대 +0.003~0.010

### ② 산식손실 MLP 강화 — **4절에서 진행 중**

| 항목 | 지금 | 바꿀 것 | 근거 |
|---|---|---|---|
| **피처** | LightGBM 중요도 상위 200 | 전체 880 / 상위 400 비교 | **트리에 맞춘 선택이 매끄러운 모델에 최적일 이유가 없다** |
| **학습량** | 61에폭 = 기울기 61스텝 | `PATIENCE` 120, `MAX_EPOCHS` 800 | full-batch라 스텝 수가 절대적으로 적다 |
| **구조** | 256-256 | 512-512, 3층 | |
| **시드·T** | 5개, T=0.006 | 10개, T 3종 앙상블 | 값싼 분산 감소 |

⚠️ 판단 지표는 **A안**.

### ③ σ 자체를 줄인다 — 1등 따라잡기의 본체
격차의 98%가 여기다. 우리 σ의 89%는 예보 오차(14절).

| 카드 | 기대 | 비용 |
|---|---|---|
| **격자 CNN 풍속 모델** ⭐ | **높음** — 850개 평평한 피처가 공간 구조를 못 살린다 | 큼 |
| 파워커브 입력에 밀도보정 풍속 | 낮음 | **거의 0 (한 줄)** — IEC 61400-12-1 |
| TI = σ/U 정규화 | 낮~중 | 작음 |
| 대기 안정도(벌크 리처드슨 수) | 중 | 중간 |

**순서**: 값싼 것(밀도보정·TI) 묶어서 한 번에 → 효과 없으면 바로 격자 CNN.

### ‼️ `train.ipynb` / `inference.ipynb` — 데드라인 2026-08-10
2차 평가 필수 요건. 성능이 아무리 좋아도 이게 없으면 무의미하다.
**`src/pipeline.py`가 생겼으므로 두 노트북은 얇은 껍데기면 된다** — 위 2절의 `build_final_parts()`를
`train.ipynb`(학습·저장)와 `inference.ipynb`(로드·예측)로 쪼개는 작업이다.